In [6]:
import wget, torch, os
from raw import load_imdb_synth, load_imdb, load_xor
from torch.utils.data import DataLoader


## Data loading and Padding

In [18]:
# ----------------------------
# DATA
# ----------------------------
(train_data, train_labels), (test_data, test_labels), (i2w, w2i), num_classes = load_imdb()

In [ ]:
# ----------------------------
# QUESTION 1: BATCHING + PADDING

import torch

def pad_batch(batch, pad_idx, sort=False, max_length=None):
    """
    Pads a batch of variable-length integer token sequences.

    Args:
        batch: list[list[int]]
            A batch of tokenized sequences, where each sequence is a list of word indices.
        pad_idx: int
            Vocabulary index of the padding token, usually w2i[".pad"].
        sort: bool
            If True, sort sequences from longest to shortest before padding.
        max_length: int | None
            If given, truncate sequences to this maximum length before padding.

    Returns:
        torch.LongTensor of shape (batch_size, max_seq_len)
    """

    if sort:
        batch = sorted(batch, key=len, reverse=True)

    if max_length is not None:
        batch = [seq[:max_length] for seq in batch]

    max_len = max(len(seq) for seq in batch)

    padded = [
        seq + [pad_idx] * (max_len - len(seq))
        for seq in batch
    ]

    return torch.tensor(padded, dtype=torch.long)


def labels_to_tensor(labels):
    """
    Converts integer class labels to a tensor suitable for CrossEntropyLoss.
    """

    return torch.tensor(labels, dtype=torch.long)

In [27]:
print("Example batch of token sequences (before padding):", train_data[:10])
print("Example batch of labels:", train_labels[:10])

Example batch of token sequences (before padding): [[14, 19, 9, 379, 22, 11, 50, 52, 53, 290], [13, 574, 25, 809, 14, 32, 63, 26, 2722, 2231, 312], [10721, 4, 10956, 129, 6, 124, 88114, 5, 6, 19, 93, 4118], [198, 351, 17697, 116, 31, 13, 80, 40, 1240, 8, 69, 272, 883, 1749], [60, 913, 366, 19, 118, 836, 44, 431, 902, 60, 286, 35, 34, 1834, 11], [15069, 398, 1415, 9, 4, 122, 398, 7, 15069, 11, 16, 62, 516, 398, 7, 15069], [14, 9, 4, 6750, 19, 320, 7, 3526, 3931, 1911, 167, 22, 43, 29, 60, 986, 388], [6, 664, 7, 129, 27, 889, 8, 2578, 89, 750, 2297, 5, 7684, 78, 14, 19, 9], [13, 90, 25, 123, 138, 13, 42, 14, 19, 40, 73, 22, 13, 116, 79, 1414, 7, 153, 11], [84, 19, 262, 4, 210, 52811, 598, 35, 240, 14, 2423, 8932, 55, 24, 31, 418, 257, 15, 310, 285]]
Example batch of labels: [1, 1, 1, 1, 1, 0, 0, 1, 0, 1]


In [30]:
# printing first 10 token sequences and convert them to words for better visualization
[[i2w[w] for w in train_data[idx]] for idx in range(5)]

[['this',
  'movie',
  'is',
  'terrible',
  'but',
  'it',
  'has',
  'some',
  'good',
  'effects'],
 ['i',
  'wouldn',
  't',
  'rent',
  'this',
  'one',
  'even',
  'on',
  'dollar',
  'rental',
  'night'],
 ['ming',
  'the',
  'merciless',
  'does',
  'a',
  'little',
  'bardwork',
  'and',
  'a',
  'movie',
  'most',
  'foul'],
 ['long',
  'boring',
  'blasphemous',
  'never',
  'have',
  'i',
  'been',
  'so',
  'glad',
  'to',
  'see',
  'ending',
  'credits',
  'roll'],
 ['no',
  'comment',
  'stupid',
  'movie',
  'acting',
  'average',
  'or',
  'worse',
  'screenplay',
  'no',
  'sense',
  'at',
  'all',
  'skip',
  'it']]

In [31]:
for i in range(5):
    print(f"Sample {i}")
    print("ids:    ", train_data[i])
    print("words:  ", [i2w[idx] for idx in train_data[i]])
    print("label:  ", train_labels[i])
    print()

Sample 0
ids:     [14, 19, 9, 379, 22, 11, 50, 52, 53, 290]
words:   ['this', 'movie', 'is', 'terrible', 'but', 'it', 'has', 'some', 'good', 'effects']
label:   1

Sample 1
ids:     [13, 574, 25, 809, 14, 32, 63, 26, 2722, 2231, 312]
words:   ['i', 'wouldn', 't', 'rent', 'this', 'one', 'even', 'on', 'dollar', 'rental', 'night']
label:   1

Sample 2
ids:     [10721, 4, 10956, 129, 6, 124, 88114, 5, 6, 19, 93, 4118]
words:   ['ming', 'the', 'merciless', 'does', 'a', 'little', 'bardwork', 'and', 'a', 'movie', 'most', 'foul']
label:   1

Sample 3
ids:     [198, 351, 17697, 116, 31, 13, 80, 40, 1240, 8, 69, 272, 883, 1749]
words:   ['long', 'boring', 'blasphemous', 'never', 'have', 'i', 'been', 'so', 'glad', 'to', 'see', 'ending', 'credits', 'roll']
label:   1

Sample 4
ids:     [60, 913, 366, 19, 118, 836, 44, 431, 902, 60, 286, 35, 34, 1834, 11]
words:   ['no', 'comment', 'stupid', 'movie', 'acting', 'average', 'or', 'worse', 'screenplay', 'no', 'sense', 'at', 'all', 'skip', 'it']
label: 

In [32]:
lengths =[len(sample) for sample in train_data]
print("Max length:", max(lengths))
print("Min length:", min(lengths))
print("Average length:", sum(lengths) / len(lengths))
print("Unique lengths:", set(lengths))

Max length: 2514
Min length: 10
Average length: 240.6318
Unique lengths: {10, 11, 12, 14, 15, 16, 17, 19, 20, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215

In [33]:
print("PAD index:", w2i[".pad"])
print("Vocabulary size:", len(i2w))
print("First 15 vocab items:", i2w[:15])

PAD index: 0
Vocabulary size: 99430
First 15 vocab items: ['.pad', '.start', '.end', '.unk', 'the', 'and', 'a', 'of', 'to', 'is', 'br', 'it', 'in', 'i', 'this']
